In [2]:
import numpy as np
import pandas as pd
print(np.__version__)
print(pd.__version__)
import jax
print(jax.__version__)
print(jax.devices())

1.26.4
2.3.3
0.6.2
[CudaDevice(id=0)]


Importing cellular automata & optimization classes, and other stuff

In [3]:
import os
import sys
import shutil

from typing import List, Type, Callable, Dict
from numpy import int32
from numpy._typing import NDArray
import importlib

sys.path.append(os.path.dirname(os.path.dirname(os.path.abspath(''))))

#from algorithm.blender import Lattice, clear_initial
from algorithm.genetic import Optimizer, Mutator, ArbitraryRulesetMutator
from algorithm.objectives import sampleobj, surface_to_vol, surfacecalc

import numpy as np
import pandas as pd

import time

Setting up optimizer and data logging code

In [ ]:
def log_mutation(data_list: List[Dict], mutations: List[tuple[List,List]], objective_val: float):
    """
    Given the data list reference, the mutation set, and the objective value after applying it, add it to the data logging list
    """
    ic_cell_pos = []
    ic_state_old = []
    ic_state_new = []
    srt_cell_pos = []
    srt_state_old = []
    srt_state_new = []

    """
    Important difference from original genetic algorithm: 
    ic_mut is a 4 membered list showing the mutated position in the IC.
    Example: [0,0,0,1], meaning that at IC pos (0,0) the state 0 is modified to be 1.
    srt_mut is a 6 membered list showing the mutated position in the ruleset.
    Example: [0,0,0,0,0,1], meaning that at SRT rule pos (0,0,0) index 0 the rule 0 is modified to be 1.
    For a 2 state SRT there are 18 different cells for mutation: 0 - 17.
    """
    for ic_mut in mutations[0]:
        ic_cell_pos.append(tuple(ic_mut[0:2]))
        ic_state_old.append(ic_mut[2])
        ic_state_new.append(ic_mut[3])
    for srt_mut in mutations[1]:
        srt_cell_pos.append(tuple(srt_mut[0:4]))
        srt_state_old.append(srt_mut[4])
        srt_state_new.append(srt_mut[5])
    # print(f"Number of IC mutations: {len(mutations[0])}, Number of SRT mutations: {len(mutations[1])}", end='\r')
    data_list.append({
        "ic_cell_pos": np.array(ic_cell_pos), 
        "ic_state_old": np.array(ic_state_old), 
        "ic_state_new": np.array(ic_state_new), 
        "srt_cell_pos": np.array(srt_cell_pos), 
        "srt_state_old": np.array(srt_state_old), 
        "srt_state_new": np.array(srt_state_new), 
        "objective": objective_val,
    })

itlogs = [0]

def run_experiment(iters: int, grid_sz: int, opt_func: Callable[[NDArray[int32]], int], ic_num_mutate: int, srt_num_mutate: int, rule_mutate_prob: float, strict: bool = False, num_strict: bool = True, states: int = 2, ic_enable: bool = True, srt_enable: bool = True):
    """
    Runs an experiment with the below hyperparameters:

    :param iters: The number of iterations the mutation algorithm (updating both IC and SRT) is going to run for
    :param grid_sz: The size of the square grid that we're going to update each iteration
    :param opt_func: The functions that gives the performance metric we're going to optimize
    :param srt_num_mutate: The number of SRT cells for which we're going to mutate the rule applied, each iteration
    :param ic_num_mutate: The number of IC cells for which we're going to mutate the rule applied, each iteration
    :param rule_mutate_prob: The probability, for each neighbor state tensor of the rule of a cell that's selected to be mutated, the final state is mutated
    :param strict: Whether SRT mutations are chosen by cell then rule or by rule directly; for more info, see mutations.py
    :param num_strict: Whether the number of SRT/IC cells mutated will be constant per iteration or variable; for more info, see mutations.py
    :param states: The number of states the NSTICA will run on; for more info, see nstica.py
    :param ic_enable: Whether the IC will be mutated
    :param srt_enable: Whether the SRT will be mutated

    Params ruleset_mutator_class and rule_set have been deleted due to previous deletion of the RulesetMutator class.
    """

    # RESOLVED: separate SRT and IC mutations to have a certain number of each
    # RESOLVED: add a flag to enable doing only SRT or only IC mutations in an iteration (in optimizer step, and then propagate into mutator)

    ruleset_mutator = ArbitraryRulesetMutator(grid_size=grid_sz, mutate_p=1/(grid_sz**2) * (srt_num_mutate+ic_num_mutate), rule_mutate_p=rule_mutate_prob, strict=strict, num_strict = num_strict, ic_ct = ic_num_mutate, srt_ct = srt_num_mutate, states=states, ic_enable=ic_enable, srt_enable = srt_enable)

    optim = Optimizer(mutator=ruleset_mutator, objective=lambda grid: opt_func(grid))
    """
    Pandas Dataframe used to log experiment data is:

    ic_cell_pos (np.array) | ic_state_old (np.array) | ic_state_new (np.array) | srt_cell_pos (np.array) | srt_state_old (np.array) | srt_state_new (np.array) | objective (float)
    
    etc.

    initial state for IC is in entry 0 in ic_state_old, and SRT is in entry 0 in srt_state_old

    ic and srt mutation cell positions and states can have an extra dimension in the beginning to indicate they are batch updates
    """

    init_state = optim.state
    
    data_list = [{"ic_cell_pos": grid_sz, 
                  "ic_state_old": init_state[0], 
                  "ic_state_new": None, 
                  "srt_cell_pos": -1, 
                  "srt_state_old": init_state[1], 
                  "srt_state_new": None, 
                  "objective": 0}]

    for it in range(iters):
        # print(f"Iteration {it}", end='\r')
        # print(f"On iteration {it+1}...")
        accepted, new, old, mutations = optim.step()
        # data logging
        log_mutation(data_list, mutations, optim.objvalue)
        #itlogs.append(float(optim.objvalue))
        #if accepted:
        #   print(f"Got a better state: {optim.objvalue} at iteration {it}, which is {(100*optim.objvalue/(-98304)):.3f}% of the ground truth.")

    # print(data_list)
    df = pd.DataFrame(data_list)
    # print(df)
    return df

timelogs = []

def repeat_experiment(experiment_name: str, num_expers: int, *args):
    """
    Perform (sequentially) multiple experiments that return a Pandas DataFrame and save all the data

    :param experiment_name: The name of the experiment to save the file
    :param num_expers: Number of times to run the experiment (and save all the data in one file)
    :param *args: The arguments to be passed to the experiment function

    Within *args should be a states parameter, denoting the number of NSTICA states.
    """
    for i in range(num_expers):
        init = time.time()
        print(f'REPETITION {i}')
        ret_data = run_experiment(*args)
        timelogs.append(time.time() - init)
        print(f"Finished rep {i} in {time.time() - init}s")
        ret_data.to_hdf(f'gpu_test/{states}/{experiment_name}_{i}_{states}states.h5', key='data', mode='a')


Setting up experiments and gathering data

In [5]:
#ITERATIONS_SET = [50, 100, 200, 500]
#GRID_SIZE_SET = [10, 20, 32, 64]
NUM_REPEAT = 20
EXPERIMENT_NAME = "statetimetest"
#STATE_SET = [2, 3, 4, 5]
#NUMS_SET = [(1, 18), (2, 36), (5, 90), (10, 180), (20, 360), (50, 900), (100, 1800), (200, 3600), (500, 9000), (1000, 18000), (2000, 36000)]
COND_SET = [(50, 10, 2), (50, 20, 2), (50, 32, 2), (50, 64, 2), (50, 100, 2), (100, 10, 2), (100, 20, 2), (100, 32, 2), (100, 64, 2), (100, 100, 2), (200, 10, 2), (200, 20, 2), (200, 32, 2), (200, 64, 2), (200, 100, 2), (500, 10, 2), (500, 20, 2), (500, 32, 2), (500, 64, 2), (500, 100, 2), (50, 10, 3), (50, 20, 3), (50, 32, 3), (50, 64, 3), (100, 10, 3), (100, 20, 3), (100, 32, 3), (200, 10, 3), (200, 20, 3), (200, 32, 3), (500, 10, 3), (500, 20, 3), (50, 10, 4), (50, 20, 4), (50, 32, 4), (100, 10, 4), (100, 20, 4), (200, 10, 4), (500, 10, 4), (50, 10, 5), (50, 20, 5), (100, 10, 5), (200, 10, 5), (50, 10, 6)]
for conditions in COND_SET:
#for iters in ITERATIONS_SET:
    #for grid_sz in GRID_SIZE_SET:
        #for states in STATE_SET:
            #for nums in NUMS_SET:
                iters = conditions[0]
                grid_sz = conditions[1]
                states = conditions[2]
                print(f"RUNNING EXPERIMENT {EXPERIMENT_NAME} WITH {iters} ITERATIONS AND {grid_sz} SIZE GRID AND {states} STATES")
                #Default probability for Strict Mode = 2/3; default probability for Non-Strict Mode = 5/384
                #Default number of IC mutations = 20, default number of SRT mutations = 240
                repeat_experiment(f"{EXPERIMENT_NAME}_{iters}ITERS_{grid_sz}GRID", NUM_REPEAT, iters, grid_sz, surfacecalc, grid_sz, grid_sz**2, 2/3, True, True, states, True, True)
                print(itlogs)

RUNNING EXPERIMENT statetimetest WITH 50 ITERATIONS AND 10 SIZE GRID AND 2 STATES
REPETITION 0
Finished rep 0 in 6.1389453411102295s


/tmp/ipykernel_451365/3661431194.py:121: PerformanceWarning: 
your performance may suffer as PyTables will pickle object types that it cannot
map directly to c-types [inferred_type->mixed-integer,key->block0_values] [items->Index(['ic_cell_pos', 'ic_state_old', 'ic_state_new', 'srt_cell_pos',
       'srt_state_old', 'srt_state_new', 'objective'],
      dtype='object')]

  ret_data.to_hdf(f'gpu_test/{states}/{experiment_name}_{i}_{states}states.h5', key='data', mode='a')


REPETITION 1
Finished rep 1 in 1.1574828624725342s
REPETITION 2


/tmp/ipykernel_451365/3661431194.py:121: PerformanceWarning: 
your performance may suffer as PyTables will pickle object types that it cannot
map directly to c-types [inferred_type->mixed-integer,key->block0_values] [items->Index(['ic_cell_pos', 'ic_state_old', 'ic_state_new', 'srt_cell_pos',
       'srt_state_old', 'srt_state_new', 'objective'],
      dtype='object')]

  ret_data.to_hdf(f'gpu_test/{states}/{experiment_name}_{i}_{states}states.h5', key='data', mode='a')


Finished rep 2 in 2.207519292831421s
REPETITION 3


...

/tmp/ipykernel_451365/3661431194.py:121: PerformanceWarning: 
your performance may suffer as PyTables will pickle object types that it cannot
map directly to c-types [inferred_type->mixed-integer,key->block0_values] [items->Index(['ic_cell_pos', 'ic_state_old', 'ic_state_new', 'srt_cell_pos',
       'srt_state_old', 'srt_state_new', 'objective'],
      dtype='object')]

  ret_data.to_hdf(f'gpu_test/{states}/{experiment_name}_{i}_{states}states.h5', key='data', mode='a')


Finished rep 18 in 10.610103368759155s
REPETITION 19


/tmp/ipykernel_451365/3661431194.py:121: PerformanceWarning: 
your performance may suffer as PyTables will pickle object types that it cannot
map directly to c-types [inferred_type->mixed-integer,key->block0_values] [items->Index(['ic_cell_pos', 'ic_state_old', 'ic_state_new', 'srt_cell_pos',
       'srt_state_old', 'srt_state_new', 'objective'],
      dtype='object')]

  ret_data.to_hdf(f'gpu_test/{states}/{experiment_name}_{i}_{states}states.h5', key='data', mode='a')


Finished rep 19 in 11.214908599853516s
[0]


/tmp/ipykernel_451365/3661431194.py:121: PerformanceWarning: 
your performance may suffer as PyTables will pickle object types that it cannot
map directly to c-types [inferred_type->mixed-integer,key->block0_values] [items->Index(['ic_cell_pos', 'ic_state_old', 'ic_state_new', 'srt_cell_pos',
       'srt_state_old', 'srt_state_new', 'objective'],
      dtype='object')]

  ret_data.to_hdf(f'gpu_test/{states}/{experiment_name}_{i}_{states}states.h5', key='data', mode='a')


In [6]:
#GPU MutVar Trial Timelogs
print(timelogs)

[6.138943195343018, 1.1574807167053223, 2.207517147064209, 2.2170217037200928, 2.104154586791992, 2.172276735305786, 2.120096445083618, 2.2206027507781982, 2.059760808944702, 2.0410895347595215, 1.9138853549957275, 2.1581695079803467, 1.9844450950622559, 2.036797285079956, 2.2244954109191895, 2.0549538135528564, 1.204390525817871, 2.185175657272339, 2.0343761444091797, 2.0435659885406494, 3.481600522994995, 2.1276097297668457, 2.068340539932251, 2.1062376499176025, 1.9962852001190186, 2.086791515350342, 2.051664352416992, 2.0382630825042725, 1.9995410442352295, 1.9592022895812988, 1.959202527999878, 1.188452959060669, 2.0478458404541016, 1.9745957851409912, 1.976597785949707, 1.9849255084991455, 2.141939401626587, 2.094918727874756, 2.0279438495635986, 2.0570147037506104, 4.12665057182312, 2.6057612895965576, 2.6394619941711426, 2.7374823093414307, 1.6218938827514648, 2.7643613815307617, 2.4558959007263184, 2.617360830307007, 2.4444966316223145, 2.5100579261779785, 2.511550188064575, 2

In [7]:
ITERATIONS_SET = [50, 100, 200, 500]
GRID_SIZE_SET = [10, 20, 32, 64]
NUM_REPEAT = 20
EXPERIMENT_NAME = "comparative" #Directly comparable to naive implementation
STATE_SET = [2]
#NUMS_SET = [(1, 18), (2, 36), (5, 90), (10, 180), (20, 360), (50, 900), (100, 1800), (200, 3600), (500, 9000), (1000, 18000), (2000, 36000)]
for iters in ITERATIONS_SET:
    for grid_sz in GRID_SIZE_SET:
        for states in STATE_SET:
            #for nums in NUMS_SET:
                print(f"RUNNING EXPERIMENT {EXPERIMENT_NAME} WITH {iters} ITERATIONS AND {grid_sz} SIZE GRID AND {states} STATES")
                #Default probability for Strict Mode = 2/3; default probability for Non-Strict Mode = 5/384
                #Default number of IC mutations = 20, default number of SRT mutations = 240
                repeat_experiment(f"{EXPERIMENT_NAME}_{iters}ITERS_{grid_sz}GRID", NUM_REPEAT, iters, grid_sz, surface_to_vol, 10, 10, 2/3, True, False, states, True, True)
                print(itlogs)

RUNNING EXPERIMENT comparative WITH 50 ITERATIONS AND 10 SIZE GRID AND 2 STATES
REPETITION 0
Finished rep 0 in 2.2440450191497803s
REPETITION 1


/tmp/ipykernel_451365/3661431194.py:121: PerformanceWarning: 
your performance may suffer as PyTables will pickle object types that it cannot
map directly to c-types [inferred_type->mixed-integer,key->block0_values] [items->Index(['ic_cell_pos', 'ic_state_old', 'ic_state_new', 'srt_cell_pos',
       'srt_state_old', 'srt_state_new', 'objective'],
      dtype='object')]

  ret_data.to_hdf(f'gpu_test/{states}/{experiment_name}_{i}_{states}states.h5', key='data', mode='a')


Finished rep 1 in 2.196260929107666s
REPETITION 2


/tmp/ipykernel_451365/3661431194.py:121: PerformanceWarning: 
your performance may suffer as PyTables will pickle object types that it cannot
map directly to c-types [inferred_type->mixed-integer,key->block0_values] [items->Index(['ic_cell_pos', 'ic_state_old', 'ic_state_new', 'srt_cell_pos',
       'srt_state_old', 'srt_state_new', 'objective'],
      dtype='object')]

  ret_data.to_hdf(f'gpu_test/{states}/{experiment_name}_{i}_{states}states.h5', key='data', mode='a')


Finished rep 2 in 2.1047306060791016s
REPETITION 3


...

REPETITION 18
Finished rep 18 in 122.89261174201965s


/tmp/ipykernel_451365/3661431194.py:121: PerformanceWarning: 
your performance may suffer as PyTables will pickle object types that it cannot
map directly to c-types [inferred_type->mixed-integer,key->block0_values] [items->Index(['ic_cell_pos', 'ic_state_old', 'ic_state_new', 'srt_cell_pos',
       'srt_state_old', 'srt_state_new', 'objective'],
      dtype='object')]

  ret_data.to_hdf(f'gpu_test/{states}/{experiment_name}_{i}_{states}states.h5', key='data', mode='a')


REPETITION 19
Finished rep 19 in 123.30591917037964s


/tmp/ipykernel_451365/3661431194.py:121: PerformanceWarning: 
your performance may suffer as PyTables will pickle object types that it cannot
map directly to c-types [inferred_type->mixed-integer,key->block0_values] [items->Index(['ic_cell_pos', 'ic_state_old', 'ic_state_new', 'srt_cell_pos',
       'srt_state_old', 'srt_state_new', 'objective'],
      dtype='object')]

  ret_data.to_hdf(f'gpu_test/{states}/{experiment_name}_{i}_{states}states.h5', key='data', mode='a')


[0]


In [8]:
#GPU Naive-Comparable Trial Timelogs
print(timelogs)

[6.138943195343018, 1.1574807167053223, 2.207517147064209, 2.2170217037200928, 2.104154586791992, 2.172276735305786, 2.120096445083618, 2.2206027507781982, 2.059760808944702, 2.0410895347595215, 1.9138853549957275, 2.1581695079803467, 1.9844450950622559, 2.036797285079956, 2.2244954109191895, 2.0549538135528564, 1.204390525817871, 2.185175657272339, 2.0343761444091797, 2.0435659885406494, 3.481600522994995, 2.1276097297668457, 2.068340539932251, 2.1062376499176025, 1.9962852001190186, 2.086791515350342, 2.051664352416992, 2.0382630825042725, 1.9995410442352295, 1.9592022895812988, 1.959202527999878, 1.188452959060669, 2.0478458404541016, 1.9745957851409912, 1.976597785949707, 1.9849255084991455, 2.141939401626587, 2.094918727874756, 2.0279438495635986, 2.0570147037506104, 4.12665057182312, 2.6057612895965576, 2.6394619941711426, 2.7374823093414307, 1.6218938827514648, 2.7643613815307617, 2.4558959007263184, 2.617360830307007, 2.4444966316223145, 2.5100579261779785, 2.511550188064575, 2